<a href="https://colab.research.google.com/github/RTE404/NLP-Resume-Screening-App/blob/feature%2Fbert-implementation/Resume_Screening_BERT_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets scikit-learn


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# DistilBERT: Smaller, faster version of BERT
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"Model loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Model loaded: distilbert-base-uncased
Vocabulary size: 30522


In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle
from tqdm import tqdm
import nltk
from nltk.stem import WordNetLemmatizer

# # Download NLTK resources
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# lemmatizer = WordNetLemmatizer()

# ===== STEP 1: Load Your Data =====
# Replace 'UpdatedResumeDataSet.csv' with your actual file
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ResumeDataset.csv')  # Adjust path if needed

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Unique categories: {df['Category'].nunique()}")

# ===== STEP 2: Clean Resume Text =====
def clean_resume(text):
    clean_text = re.sub(r'http\S+', '', text)
    clean_text = re.sub(r'RT|cc', '', clean_text)
    clean_text = re.sub(r'@\S+', '', clean_text)
    clean_text = re.sub(r'#\S+', '', clean_text)
    # Remove special chars but KEEP the words
    clean_text = re.sub(r'[^\w\s]', ' ', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text.lower()
    # DON'T lemmatize - BERT needs raw text!

# Apply cleaning
df['cleaned_resume'] = df['Resume'].apply(clean_resume)

# Encode labels
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['Category'])

print(f"\nCategory mapping:")
for idx, category in enumerate(le.classes_):
    print(f"{idx}: {category}")

# ===== STEP 3: Generate BERT Embeddings =====
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"\nUsing device: {device}")

def get_bert_embedding(text, max_length=512):
    """
    Extract BERT embedding for a resume.
    Returns a 768-dimensional vector (the [CLS] token representation)
    """
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding=True
    ).to(device)

    # Get embeddings (no gradients needed for inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract [CLS] token embedding (represents the whole sentence)
    cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    return cls_embedding.squeeze()

# Generate embeddings for all resumes (this takes a few minutes)
print("\nGenerating BERT embeddings (this may take 5-10 minutes)...")
embeddings = []
for idx, resume in enumerate(tqdm(df['cleaned_resume'], desc="Embedding progress")):
    embedding = get_bert_embedding(resume)
    embeddings.append(embedding)

X = np.array(embeddings)
y = df['category_encoded'].values

print(f"Embeddings shape: {X.shape}")  # Should be (num_resumes, 768)

# ===== STEP 4: Train-Test Split =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# ===== STEP 5: Train Logistic Classifier =====
print("\nTraining Logistic Regression classifier on BERT embeddings...")
clf = LogisticRegression(max_iter=1000, random_state=42, C=0.1)
clf.fit(X_train, y_train)


# ===== STEP 6: Evaluate =====
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ===== STEP 7: Save Models =====
pickle.dump(clf, open('clf_bert.pkl', 'wb'))
pickle.dump(le, open('encoder_bert.pkl', 'wb'))
pickle.dump(tokenizer, open('tokenizer_bert.pkl', 'wb'))
pickle.dump(model, open('model_bert.pkl', 'wb'))

print("\n✅ Models saved:")
print("- clf_bert.pkl (Classifier)")
print("- encoder_bert.pkl (Label Encoder)")
print("- tokenizer_bert.pkl (Tokenizer)")
print("- model_bert.pkl (BERT Model)")


Dataset shape: (962, 2)
Columns: ['Category', 'Resume']
Unique categories: 25

Category mapping:
0: Advocate
1: Arts
2: Automation Testing
3: Blockchain
4: Business Analyst
5: Civil Engineer
6: Data Science
7: Database
8: DevOps Engineer
9: DotNet Developer
10: ETL Developer
11: Electrical Engineering
12: HR
13: Hadoop
14: Health and fitness
15: Java Developer
16: Mechanical Engineer
17: Network Security Engineer
18: Operations Manager
19: PMO
20: Python Developer
21: SAP Developer
22: Sales
23: Testing
24: Web Designing

Using device: cuda

Generating BERT embeddings (this may take 5-10 minutes)...


Embedding progress: 100%|██████████| 962/962 [00:13<00:00, 73.94it/s]


Embeddings shape: (962, 768)

Training set size: (769, 768)
Test set size: (193, 768)

Training Logistic Regression classifier on BERT embeddings...

✅ Test Accuracy: 0.9067

Classification Report:
                           precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00         4
                     Arts       0.70      1.00      0.82         7
       Automation Testing       1.00      0.40      0.57         5
               Blockchain       1.00      1.00      1.00         8
         Business Analyst       1.00      1.00      1.00         6
           Civil Engineer       1.00      1.00      1.00         5
             Data Science       0.83      0.62      0.71         8
                 Database       0.86      0.86      0.86         7
          DevOps Engineer       1.00      0.91      0.95        11
         DotNet Developer       1.00      0.20      0.33         5
            ETL Developer       0.89      1.00      0.94        